# 💳 End-to-End Financial Fraud Detection & Analytics

## Executive Overview
Financial fraud detection is a critical real-time application of machine learning in banking and online payments. The key challenge in fraud detection stems from **extreme class imbalance**: fraudulent transactions typically make up less than 0.5% of total transactions.

Standard accuracy is highly misleading here — a trivial model that predicts *Genuine* for every transaction will achieve over 99.8% accuracy while catching zero fraud cases.

### Key Objectives
1. **Clean & Validate Data**: Handle duplicate records, missing values, and verify features.
2. **Address Class Imbalance**: Implement and compare **Random Undersampling**, **Random Oversampling**, **SMOTE** (Synthetic Minority Over-sampling Technique), and **Class Weighting**.
3. **Prevent Data Leakage**: Fit all feature scalers and resampling algorithms strictly on training splits.
4. **Comprehensive Evaluation**: Evaluate models using domain-relevant metrics — **Precision**, **Recall**, **F1-Score**, **ROC-AUC**, and **PR-AUC**.
5. **Threshold Optimization**: Fine-tune decision thresholds to optimize the business trade-off between false negatives (uncaught fraud) and false positives (customer friction).

## 1. Setup & Environment
We import reusable modules from `src/` to maintain clean, production-ready notebook code.

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), "..", "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_preprocessing import load_data, clean_data, preprocess_pipeline
from eda import (
    basic_info,
    target_distribution,
    plot_class_distribution,
    plot_amount_distribution,
    plot_correlation_heatmap,
    plot_boxplot_amount,
)
from train import run_experiments, save_model
from evaluate import (
    get_confusion_matrix,
    plot_confusion_matrix,
    plot_pr_curve,
    plot_roc_curve,
    threshold_analysis,
    feature_importance,
    plot_feature_importance,
)

## 2. Load & Clean Dataset

In [ ]:
data_path = "../data/creditcard.csv"
df = clean_data(load_data(data_path))
info = basic_info(df)
print(f"Dataset Rows: {info['rows']:,}")
print(f"Dataset Columns: {info['cols']}")
print(f"Missing Values: {info['missing_values']}")
print(f"Duplicates: {info['duplicates']}")

## 3. Target Distribution & Class Imbalance Analysis

In [ ]:
dist = target_distribution(df)
print(f"Genuine Transactions (0): {dist['genuine']:,}")
print(f"Fraud Transactions   (1): {dist['fraud']:,}")
print(f"Fraud Ratio          : {dist['fraud_pct']:.4f}%")

fig_class = plot_class_distribution(df)
plt.show()

## 4. Exploratory Data Analysis (EDA)
Analyzing transaction amounts, temporal distributions, and feature correlations.

In [ ]:
fig_amt = plot_amount_distribution(df)
plt.show()

In [ ]:
fig_box = plot_boxplot_amount(df)
plt.show()

In [ ]:
fig_corr = plot_correlation_heatmap(df)
plt.show()

## 5. Model Training & Comparison Across Imbalance Handling Strategies
We train and compare 4 distinct machine learning algorithms (**Logistic Regression**, **Decision Tree**, **Random Forest**, **XGBoost**) across 4 imbalance techniques:
1. **Baseline** (Unsampled original split)
2. **Random Undersampling** (Downsample majority class)
3. **Random Oversampling** (Duplicate minority class)
4. **SMOTE** (Synthetic Minority Over-sampling Technique via linear interpolation)
5. **Class Weighting** (`class_weight="balanced"` penalty weighting during loss computation)

In [ ]:
comparison_df, best_model, best_name, splits = run_experiments(data_path, verbose=True)
X_train, X_test, y_train, y_test, scaler = splits
comparison_df

## 6. Comprehensive Best Model Evaluation
We evaluate our top-performing model on unseen test data using detailed confusion matrices, Precision-Recall curves, and ROC curves.

In [ ]:
print(f"Selected Best Model: {best_name}")
cm = get_confusion_matrix(best_model, X_test, y_test)
print(f"True Positives (Caught Fraud) : {cm['tp']}")
print(f"False Negatives (Missed Fraud): {cm['fn']}")
print(f"False Positives (False Alarm) : {cm['fp']}")
print(f"True Negatives (Genuine OK)   : {cm['tn']}")

fig_cm = plot_confusion_matrix(cm["matrix"])
plt.show()

In [ ]:
fig_pr = plot_pr_curve(best_model, X_test, y_test)
plt.show()

In [ ]:
fig_roc = plot_roc_curve(best_model, X_test, y_test)
plt.show()

## 7. Decision Threshold Optimization
By default, classifiers use a probability threshold of 0.5. In financial fraud, **False Negatives are exponentially more costly** than False Positives. Adjusting the threshold allows risk operators to prioritize high recall.

In [ ]:
thresh_df = threshold_analysis(best_model, X_test, y_test)
thresh_df

## 8. Feature Importance Analysis

In [ ]:
fi_df = feature_importance(best_model, list(X_test.columns), top_n=15)
if not fi_df.empty:
    fig_fi = plot_feature_importance(fi_df, top_n=10)
    plt.show()
fi_df

## 9. Model Persistence
Save the best trained model bundle (`model` + `scaler`) and model comparison metrics to `models/`.

In [ ]:
os.makedirs("../models", exist_ok=True)
save_model(best_model, scaler, "../models/best_model.pkl")
comparison_df.to_csv("../models/model_comparison.csv")
print("Model and comparison table successfully saved!")